In [ ]:
#| hide
%load_ext aimagics

# core

This page is rather short, as there are no public facing exports.

You can get help on the usage of `%ai` and `%%ai` with

In [ ]:
%ai?

Docstring:
Prompt an LLM as magic command.

- Can be used as 
  %ai prompt
- or as 
  %%ai prompt
  code 
  and other text in the cell
File:      ~/Documents/small-projects/aimagics-repo/aimagics/aimagics/core.py

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from IPython import get_ipython
from IPython.core.magic import magics_class, Magics, line_cell_magic
import nbformat, ipynbname
from traitlets import Unicode
from aidialog.msg_parts import Text, Msg
from fastllm.chat import StreamAccum, acomplete

In [ ]:
#| exporti
@magics_class
class AIMagics(Magics):
    """AIMagics class defining %ai line and %%ai cell magic. 
    
    Can be configured with
    %config
    %config AIMagis
    %config.model = "openrouter/google/gemini-3.8-flash"
    """

    # The config values
    model = Unicode(
        "openrouter/openai/gpt-oss-120b",
        help="Provider/model to be used."
    ).tag(config=True)

    system_prompt = Unicode(
        """You are a helpful assistant living inside a user's Jupyter notebook. 
        Use markdown syntax for styling your responses.
        Keep your responses brief and to the point.\n""",
        help="The system prompt prepended to any prompt and context."
    ).tag(config=True)


    @line_cell_magic
    async def ai(self, line, cell=None):
        """Prompt an LLM as magic command.
        
        - Can be used as 
          %ai prompt
        - or as 
          %%ai prompt
          code 
          and any other text in the cell
        """
        line = line or ""
        prompt = line if cell is None else f"{line}\n{cell}".strip()

        ctx = get_context()
        final_prompt = f"{self.system_prompt}\n{ctx}\n{prompt}"

        msg = Msg('user', [Text(final_prompt)]) # TODO correct and better message history, TODO handle system prompt correctly, TODO tools
        rs = await acomplete([msg], model=self.model, stream=True)
        fmt = await adisplay_stream_own(rs)

In [ ]:
#| exporti
def line_magic_quotes(lines):
    """For line magic, add quotes if a ? is present. This prevents triggering ipython's help with ?"""
    magic_string = "%ai"
    if not lines or not lines[0].lstrip().split()[0] == magic_string: return lines
    arg = lines[0].lstrip()[len(magic_string):].strip() # the part after %ai stripped from whitespaces
    if "?" in arg:
        arg = arg.strip("'\"") # remove any " or ' at beginning or end
        line = f"{magic_string} '{arg}'\n" # build up new line magic with quoted input
        lines[0] = line 
        return lines
    else: 
        return lines

In [ ]:
#| exporti
def cell_magic_dummy_character(lines):
    """For cell magic (%%ai), don't need to add quotes, 
    but need to add a dummy character in case cell is empty"""
    magic_string = "%%ai"
    if not lines or not lines[0].lstrip().split()[0] == magic_string: return lines
    if len(lines) == 1:
        lines.append("\n")
    return lines

In [ ]:
#| exporti
def get_context():
    """Get notebook cells up to and including the calling cell.

    - Relies on state of notebook on disk. Recommended to turn autosave on.
    - If multiple cells have the same source, cells only up and including the first one are included.
    - Currently just uses the raw text content (json) of the current ipynb notebook."""
    ip = get_ipython()
    source_of_calling_cell = ip.kernel.get_parent()['content']['code']
    def normalize(s):
        return s.strip().replace("'", "").replace('"', "") # becomes fuzzy matching basically
    normalized_source = normalize(source_of_calling_cell)
    nb = nbformat.read(ipynbname.path(), as_version=4)
    for i, cell in enumerate(nb['cells']):
        # there may be an error matching cells that have been modified by line_magic_quotes / cell_magic_dummy_character
        if cell['source'] == source_of_calling_cell or normalize(cell['source']) == normalized_source:
            break
    ctx = "\n".join([repr(cell) for cell in nb['cells'][:i+1]])
    return ctx

In [ ]:
#| exporti
async def adisplay_stream_own(rs, chat=None, mx=2000):
    "Use IPython.display to markdown display the response stream, refreshing from `chat.full()` on `Refresh`."
    try: from IPython.display import display, Markdown
    except ModuleNotFoundError: raise ModuleNotFoundError("This function requires ipython. Please run `pip install ipython` to use.")
    acc = StreamAccum(chat, mx=mx)
    h = display(Markdown('Processing request ...'), display_id=True) # Placeholder text
    async for o in rs:
        if acc(o) and acc.txt: 
            h.update(Markdown(acc.txt))
    return acc.txt

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()